# 🇮🇳 Bharat Praman — Serve Fine-Tuned Qwen 3B Cypher Model (Colab T4 GPU)

This notebook:
1. Unzips your fine-tuned `bharat-praman-cypher-lora.zip` LoRA adapter.
2. Loads base model `Qwen/Qwen2.5-Coder-3B-Instruct` on T4 GPU (FP16 / 4-bit) and attaches your adapter.
3. Serves an OpenAI-compatible FastAPI endpoint `/v1/chat/completions` + `/generate_plan`.
4. Uses **ngrok** (or localtunnel) to give you a public URL.
5. In your local Bharat Praman `.env`, set:
   ```env
   LOCAL_CYPHER_MODEL_URL=https://your-tunnel-url.ngrok-free.app
   ```
   Now **your fine-tuned model answers first** for all politician Cypher queries before Gemini!


In [ ]:
# 1. Install dependencies
!pip install -q -U torch transformers peft accelerate bitsandbytes fastapi uvicorn pyngrok pydantic
import torch
print("CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2. Upload or Unzip bharat-praman-cypher-lora.zip
import os, zipfile
from pathlib import Path

zip_path = "bharat-praman-cypher-lora.zip"

if not os.path.exists(zip_path):
    print("Please upload bharat-praman-cypher-lora.zip:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".zip"):
            zip_path = fn
            break

print("Extracting", zip_path, "...")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall("lora_adapter")
print("Unzipped adapter into ./lora_adapter")
!ls -lh lora_adapter

In [ ]:
# 3. Load Qwen2.5-Coder-3B + LoRA Adapter on GPU
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
ADAPTER_PATH = "./lora_adapter"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

print("Loading base model in FP16...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Fine-tuned Qwen 3B Cypher model is ready in GPU memory!")

In [ ]:
# 4. Test Local Cypher Generation
import re, json

def generate_cypher_plan(question: str):
    prompt = f"<|im_start|>system\nYou translate questions into read-only Neo4j Cypher queries for the Bharat Praman graph.<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.01,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    raw_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return raw_output

sample_q = "Which MPs have pending criminal cases?"
print("Test Question:", sample_q)
print("Model Output:\n", generate_cypher_plan(sample_q))

In [ ]:
# 5. Build Lightweight FastAPI Server
import json, re, asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Bharat Praman Fine-Tuned Cypher Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class PlanRequest(BaseModel):
    question: str

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    messages: list[ChatMessage]
    temperature: float = 0.0

@app.get("/health")
def health():
    return {"status": "ok", "model": "Qwen2.5-Coder-3B-Instruct-LoRA"}

@app.post("/generate_plan")
def plan_endpoint(req: PlanRequest):
    out = generate_cypher_plan(req.question)
    try:
        # If output is JSON
        data = json.loads(out)
        return data
    except Exception:
        # Wrap raw cypher string or JSON block
        m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", out, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                pass
        m_cypher = re.search(r"(MATCH\s+.*)", out, re.DOTALL | re.I)
        cypher_str = m_cypher.group(1).strip() if m_cypher else out
        return {
            "result_type": "graph",
            "cypher": cypher_str,
            "parameters": {},
            "explanation": "Generated by fine-tuned Qwen 3B Cypher model"
        }

@app.post("/v1/chat/completions")
def chat_endpoint(req: ChatCompletionRequest):
    user_q = req.messages[-1].content
    plan = plan_endpoint(PlanRequest(question=user_q))
    return {
        "id": "chatcmpl-qwen3b",
        "object": "chat.completion",
        "choices": [{
            "message": {
                "role": "assistant",
                "content": json.dumps(plan)
            },
            "finish_reason": "stop"
        }]
    }

In [ ]:
# 6. Expose Server via ngrok & Run
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from getpass import getpass

nest_asyncio.apply()

NGROK_TOKEN = input("Enter ngrok auth token (from dashboard.ngrok.com/get-started/your-authtoken) or press Enter for free tunnel: ").strip()
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8005).public_url
print("\n" + "="*70)
print("🚀 YOUR FINETUNED MODEL IS LIVE AT:")
print("   ", public_url)
print("="*70)
print(f"\nAdd this line to your local BACKEND/.env:\nLOCAL_CYPHER_MODEL_URL={public_url}\n")

uvicorn.run(app, host="0.0.0.0", port=8005)